# CrowdSense ML - Notebook 03: CSRNet Training on ShanghaiTech

This notebook trains **CSRNet (Congested Scene Recognition Network)**.

### Architecture Overview:
1. **Frontend**: First 10 layers of VGG-16 pretrained on ImageNet (downsampling factor = 8).
2. **Backend**: 6 dilated convolutional layers (dilation rate $r=2$) to expand receptive field from $3\times 3$ to effective $5\times 5$, capturing wide spatial context without losing resolution through pooling.
3. **Loss Function**: Euclidean Mean Squared Error (MSE) between predicted density map $\hat{D}$ and ground truth $D$:
$$L(\Theta) = \frac{1}{2N} \sum_{i=1}^{N} \|\hat{D}(X_i; \Theta) - D_i\|_2^2$$

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class CSRNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.frontend = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(True)
        )
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, dilation=2, padding=2), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, dilation=2, padding=2), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, dilation=2, padding=2), nn.ReLU(True),
            nn.Conv2d(512, 256, 3, dilation=2, padding=2), nn.ReLU(True),
            nn.Conv2d(256, 128, 3, dilation=2, padding=2), nn.ReLU(True),
            nn.Conv2d(128, 64, 3, dilation=2, padding=2), nn.ReLU(True)
        )
        self.output_layer = nn.Conv2d(64, 1, 1)
    
    def forward(self, x):
        return torch.relu(self.output_layer(self.backend(self.frontend(x))))

model = CSRNet()
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")